In [4]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [6]:
loader = TextLoader('langchain_crewai_dataset.txt')
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size= 200, chunk_overlap= 20)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs,'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple'),
 Document(metad

In [9]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [10]:
retriever=vectorstore.as_retriever(search_type='mmr', search_kwargs= {"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000014110BCC1A0>, search_type='mmr', search_kwargs={'k': 5})

In [12]:
## Here we are adding LLM and Prompt for Query Enhancement
import os
from dotenv import load_dotenv
load_dotenv


llm = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai"
)


In [22]:
query_expansion_prompt_template = PromptTemplate.from_template(
    """You are a helpful assistant. Expand the following query to improve the document retrieval by adding by adding relevant synonyms, technical terms, and useful context


    query = "{query}"
    
    Expanded Query
    """
)

In [23]:
quer_expansion_chain = query_expansion_prompt_template|llm| StrOutputParser()
quer_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='You are a helpful assistant. Expand the following query to improve the document retrieval by adding by adding relevant synonyms, technical terms, and useful context\n\n\n    query = "{query}"\n\n    Expanded Query\n    ')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001410FF3E450>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000014136473E90>, root_client=<openai.OpenAI object at 0x0000014112C82300>, root_async_client=<openai.AsyncOpenAI object at 0x0000014136472690>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)
| StrOutputParser()

In [24]:
quer_expansion_chain.invoke({"query" : "langchain memory"})

'To enhance the document retrieval for the query "langchain memory," we can expand it by including relevant synonyms, technical terms, and useful context related to LangChain\'s memory capabilities and functionalities. Here\'s an expanded version of the query:\n\n**Expanded Query:**\n\n"langchain memory management, LangChain persistent storage, LangChain state management, LangChain context retention, memory integration in LangChain, LangChain vector store, memory optimization, conversational memory in LangChain, knowledge management, memory capabilities in LangChain, AI memory systems, distributed memory architectures, persistent memory in conversational agents, LangChain retrievable memory, long-term memory in AI models, memory caching, LangChain embeddings, context-aware memory in AI."\n\nThis expanded query incorporates various synonyms and related technical terms that can help in uncovering a broader range of documents about LangChain\'s memory functionalities.'

In [25]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [26]:
rag_pipeline=(
    RunnableMap({
        "input" :lambda x:x["input"],
        "context" : lambda x: retriever.invoke(quer_expansion_chain.invoke({"query":x["input"]}))

        })
        | document_chain
)

In [28]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(quer_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

To enhance the document retrieval for the query regarding the types of memory supported by LangChain, we can incorporate relevant synonyms, technical terms, and contextual information. Here’s an expanded version of the query:

```json
{
  "input": "What types of memory architectures or memory management techniques does LangChain support? Please provide information on different memory models, such as persistent memory, episodic memory, vector storage, and any caching strategies. Additionally, include details about how LangChain integrates memory with its components, including agent memory, user memory, and external memory interfaces. I am also interested in comparing the memory functionalities with other similar frameworks."
}
```

### Breakdown of the Expansion:

1. **Synonyms and Variants**: 
   - “memory architectures” and “memory management techniques” provide broader terms.
   - “persistent memory”, “episodic memory”, and “vector storage” specify types of memory.

2. **Technical Te